<a href="https://colab.research.google.com/github/ljzier/ST-554-repo/blob/main/Zier_ST_554_HW10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Linda Zier**

**ST 554**

**HW #10**

## The Goal

The goal of this assignment is to get practice using spark structured streaming to deal with data.

## Part 1 - Creating Streaming Data Using rate
First we setup a data stream using the "rate" format.
Prior to starting the stream, we set up a sequence of actions that used the rate data to find the square root of the rate ‘value’ and to find mod 4 of the rate ‘value’.

We outputted this using a writeStream that wrote to ‘memory’ (format("memory")).
We ran it for 30 seconds and then stopped the query. The entire output table is shown below.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

spark = SparkSession.builder.getOrCreate()

# set up data stream with rate format
rateDF = spark.readStream.format("rate").load()

# find the sqrt of "value"
rateDF = rateDF.withColumn("sqrt_value", sqrt(col("value")))

# find mod 4 of "value"
rateDF = rateDF.withColumn("mod_value", col("value") % 4)

# output to memory
writeDF = rateDF.writeStream.outputMode("append").format("memory").queryName("rate_query").start()

# run for 30 seconds
time.sleep(30)

#stop query
writeDF.stop()

# output table
spark.sql("select * from rate_query").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 08:17:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/20 08:17:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/20 08:17:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c56c96da-3278-4394-abd1-2543b4d6153b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 08:17:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/20 08:17:45 

+--------------------+-----+------------------+---------+
|           timestamp|value|        sqrt_value|mod_value|
+--------------------+-----+------------------+---------+
|2026-04-20 08:17:...|    0|               0.0|        0|
|2026-04-20 08:17:...|    1|               1.0|        1|
|2026-04-20 08:17:...|    2|1.4142135623730951|        2|
|2026-04-20 08:17:...|    3|1.7320508075688772|        3|
|2026-04-20 08:17:...|    4|               2.0|        0|
|2026-04-20 08:17:...|    5|  2.23606797749979|        1|
|2026-04-20 08:17:...|    6| 2.449489742783178|        2|
|2026-04-20 08:17:...|    7|2.6457513110645907|        3|
|2026-04-20 08:17:...|    8|2.8284271247461903|        0|
|2026-04-20 08:17:...|    9|               3.0|        1|
|2026-04-20 08:17:...|   10|3.1622776601683795|        2|
|2026-04-20 08:17:...|   11|   3.3166247903554|        3|
|2026-04-20 08:17:...|   12|3.4641016151377544|        0|
|2026-04-20 08:17:...|   13| 3.605551275463989|        1|
|2026-04-20 08

## Part 2 - Using data from a CSV with a Pipeline

We used six bikeDetails sub datasets for this assignment. The one named bikeDetails_for_fit.csv was read in as a spark (SQL) data frame. With this spark SQL data frame we used an SQLTransformer to do some log transforms, rename a variable, and create a dummy variable from categorical variable.

```
SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
```
We also used a VectorAssembler to create a features column. The features column included the year, log_km_driven, and one_owner variables.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.getOrCreate()

# read in bikeDetails_for_fit.csv as spqrk SQL dataframe
bikeDF=spark.read.csv("data/bikeDetails_for_fit.csv", header = True, inferSchema=True)

# SQLTransformer
sqlTrans= SQLTransformer(statement =
                         '''
                         SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
                         CASE WHEN owner = "1st owner" THEN 1 ELSE 0 END AS one_owner
                         FROM __THIS__
                         ''')

#VectorAssembler to bundle features together
assembler = VectorAssembler(
    inputCols=['year', 'log_km_driven', 'one_owner'],
    outputCol='features')

print("transformations complete")

transformations complete



We then created a Pipeline with the two steps above (SQLTransformer then VectorAssembler) and fit this pipeline to the SQL data frame.

In [ ]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sqlTrans, assembler])
fitted_pipeline = pipeline.fit(bikeDF)

print("pipeline complete")


pipeline complete


Lastly we set up a read stream to look for csv files placed into a folder (bike_data in my case). When a csv came in, we transformed it using the fitted pipeline’s .transform() method. We used the SQL data frame's schema and also noted that the incoming data had a header.


In [ ]:
# setup a read stream
bikeStream = spark.readStream.schema(bikeDF.schema).option("header", True).csv("bike_data")

# transform new data
transStream = fitted_pipeline.transform(bikeStream)



We placed the other files into the (bike_data) folder one at a time to simulate streaming data.  The output appears below.


In [ ]:
# write the transformed bike stream to the console
query=transStream.writeStream.outputMode("append").format("console").start()

26/04/20 08:17:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b78b6da3-c7af-4e18-9c5f-94fa30240f70. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 08:17:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
# stop the query
input("Press Enter to stop query")
query.stop()
print("Query stopped")

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

Press Enter to stop query 


Query stopped


26/04/20 08:20:34 WARN DAGScheduler: Failed to cancel job group 38042982-570c-4e86-a79c-e0f1dc7cd515. Cannot find active jobs for it.
26/04/20 08:20:34 WARN DAGScheduler: Failed to cancel job group 38042982-570c-4e86-a79c-e0f1dc7cd515. Cannot find active jobs for it.
